# Hello RNN — Training a Vanilla RNN on "hello"

We train a vanilla RNN with **3 hidden units** (Karpathy's architecture) on the sequence `hello`.

**Vocabulary:** `h=0, e=1, l=2, o=3`

**Training pairs** (input → target):
- `h` → `e`
- `e` → `l`
- `l` → `l`  ← this is the hard one
- `l` → `o`

**Forward pass:**
$$h^{(t)} = \tanh(W_{xh}\,x^{(t)} + W_{hh}\,h^{(t-1)})$$
$$\hat{y}^{(t)} = W_{hy}\,h^{(t)}$$

Loss is cross-entropy over softmax of the logits, summed over all 4 timesteps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Vocabulary
chars  = ['h', 'e', 'l', 'o']
vocab  = {c: i for i, c in enumerate(chars)}
VOCAB  = len(chars)  # 4
HIDDEN = 3           # Karpathy's architecture

def one_hot(i, size=VOCAB):
    v = np.zeros(size)
    v[i] = 1.0
    return v

# Training sequence
sequence = 'hello'
xs = [one_hot(vocab[c]) for c in sequence[:-1]]  # h e l l
ts = [vocab[c]          for c in sequence[1:]]   # e l l o (as indices)

print('Inputs: ', list(sequence[:-1]))
print('Targets:', list(sequence[1:]))

## Initialise Weights

- `W_xh`: shape `(HIDDEN, VOCAB)` = `(3, 4)` — input to hidden
- `W_hh`: shape `(HIDDEN, HIDDEN)` = `(3, 3)` — hidden to hidden (recurrent)
- `W_hy`: shape `(VOCAB, HIDDEN)` = `(4, 3)` — hidden to output

In [ ]:
def init_weights():
    W_xh = np.random.randn(HIDDEN, VOCAB)  * 0.01
    W_hh = np.random.randn(HIDDEN, HIDDEN) * 0.01
    W_hy = np.random.randn(VOCAB,  HIDDEN) * 0.01
    return W_xh, W_hh, W_hy

W_xh, W_hh, W_hy = init_weights()
print('W_xh shape:', W_xh.shape)
print('W_hh shape:', W_hh.shape)
print('W_hy shape:', W_hy.shape)

## Forward Pass

In [ ]:
def softmax(z):
    e = np.exp(z - z.max())  # subtract max for numerical stability
    return e / e.sum()

def forward(W_xh, W_hh, W_hy, xs, h0=None):
    h = np.zeros(HIDDEN) if h0 is None else h0.copy()
    hs, ps = [], []
    loss = 0.0

    for x, t in zip(xs, ts):
        h = np.tanh(W_xh @ x + W_hh @ h)
        p = softmax(W_hy @ h)
        loss += -np.log(p[t] + 1e-12)
        hs.append(h.copy())
        ps.append(p.copy())

    return loss, hs, ps

## Backward Pass (BPTT)

Backpropagation through time — unroll 4 steps and compute gradients analytically.

In [ ]:
def backward(W_xh, W_hh, W_hy, xs, hs, ps):
    dW_xh = np.zeros_like(W_xh)
    dW_hh = np.zeros_like(W_hh)
    dW_hy = np.zeros_like(W_hy)
    dh_next = np.zeros(HIDDEN)

    for t in reversed(range(len(xs))):
        # softmax cross-entropy gradient
        dy = ps[t].copy()
        dy[ts[t]] -= 1.0

        dW_hy += np.outer(dy, hs[t])

        # backprop into hidden state
        dh = W_hy.T @ dy + dh_next

        # backprop through tanh: d/dx tanh(x) = 1 - tanh(x)^2
        dtanh = (1 - hs[t] ** 2) * dh

        dW_xh += np.outer(dtanh, xs[t])
        h_prev = hs[t - 1] if t > 0 else np.zeros(HIDDEN)
        dW_hh += np.outer(dtanh, h_prev)

        dh_next = W_hh.T @ dtanh

    # clip gradients to prevent explosion
    for g in [dW_xh, dW_hh, dW_hy]:
        np.clip(g, -5, 5, out=g)

    return dW_xh, dW_hh, dW_hy

## Training Loop

In [ ]:
W_xh, W_hh, W_hy = init_weights()

lr       = 0.1
n_epochs = 2000
losses   = []

for epoch in range(n_epochs):
    loss, hs, ps = forward(W_xh, W_hh, W_hy, xs)
    dW_xh, dW_hh, dW_hy = backward(W_xh, W_hh, W_hy, xs, hs, ps)

    W_xh -= lr * dW_xh
    W_hh -= lr * dW_hh
    W_hy -= lr * dW_hy

    losses.append(loss)

    if epoch % 200 == 0:
        print(f'Epoch {epoch:4d}  loss={loss:.4f}')

print(f'\nFinal loss: {losses[-1]:.4f}')

## Plot Training Loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Training loss — Hello RNN (3 hidden units)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Evaluate Predictions

In [ ]:
loss, hs, ps = forward(W_xh, W_hh, W_hy, xs)

print(f"{'Input':<8} {'Target':<8} {'Pred':<8} {'OK?':<6} {'h e l o probabilities'}")
print('-' * 55)
for i, (x, t) in enumerate(zip(xs, ts)):
    pred = np.argmax(ps[i])
    ok   = '✓' if pred == t else '✗'
    prob_str = '  '.join(f'{c}:{ps[i][j]:.4f}' for j, c in enumerate(chars))
    print(f"{chars[np.argmax(x)]:<8} {chars[t]:<8} {chars[pred]:<8} {ok:<6} {prob_str}")

## Inspect the Learned Weights

In [ ]:
np.set_printoptions(precision=3, suppress=True)

print('W_xh  (3×4) — input to hidden:')
print(W_xh)
print()
print('W_hh  (3×3) — hidden to hidden (recurrent):')
print(W_hh)
print()
print('W_hy  (4×3) — hidden to output:')
print(W_hy)

## Visualise Hidden States

The key insight: the hidden state after the **first `l`** (context `hel`) should differ from after the **second `l`** (context `hell`). This is what lets the network make different predictions for the same input character.

In [ ]:
print('\nHidden states:')
for i, c in enumerate(sequence[:-1]):
    print(f'  After {repr(c)} (t={i+1}): {np.round(hs[i], 3)}')

diff = np.linalg.norm(np.array(hs[2]) - np.array(hs[3]))
print(f'\n||h(first l) - h(second l)|| = {diff:.4f}  '
      f'({"different ✓" if diff > 0.01 else "same ✗"})')